In [4]:
import pandas as pd
import numpy as np
import psycopg2

# 1. Cargar Spotify desde el archivo CSV
spotify_df = pd.read_csv('../data/raw/spotify_dataset.csv')

# 2. Conectar a PostgreSQL y extraer la tabla cruda raw_grammys
conn = psycopg2.connect(
    dbname="music_dw",
    user="postgres",
    password="postgres",
    host="localhost",
    port="5432"
)

grammy_df = pd.read_sql_query("SELECT * FROM raw_grammys;", conn)
conn.close()

print("✅ Fuentes cargadas correctamente.")
print(f"Spotify: {spotify_df.shape[0]:,} filas y {spotify_df.shape[1]} columnas")
print(f"Grammy:  {grammy_df.shape[0]:,} filas y {grammy_df.shape[1]} columnas")

✅ Fuentes cargadas correctamente.
Spotify: 114,000 filas y 21 columnas
Grammy:  4,810 filas y 10 columnas


C:\Users\Usuario\AppData\Local\Temp\ipykernel_7564\482104333.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  grammy_df = pd.read_sql_query("SELECT * FROM raw_grammys;", conn)


In [5]:
print("=== 1. ESTRUCTURA Y COMPLETITUD: SPOTIFY ===")
spotify_summary = pd.DataFrame({
    'Data Type': spotify_df.dtypes,
    'Null Count': spotify_df.isnull().sum(),
    'Null Percentage (%)': (spotify_df.isnull().sum() / len(spotify_df)) * 100
})
print(spotify_summary[spotify_summary['Null Count'] > 0])
if spotify_summary['Null Count'].sum() == 0:
    print("No se encontraron valores nulos en Spotify (salvo análisis columna por columna).")

print("\n" + "="*60 + "\n")

print("=== 1. ESTRUCTURA Y COMPLETITUD: GRAMMY ===")
grammy_summary = pd.DataFrame({
    'Data Type': grammy_df.dtypes,
    'Null Count': grammy_df.isnull().sum(),
    'Null Percentage (%)': (grammy_df.isnull().sum() / len(grammy_df)) * 100
})
print(grammy_summary[grammy_summary['Null Count'] > 0])

=== 1. ESTRUCTURA Y COMPLETITUD: SPOTIFY ===
           Data Type  Null Count  Null Percentage (%)
artists       object           1             0.000877
album_name    object           1             0.000877
track_name    object           1             0.000877


=== 1. ESTRUCTURA Y COMPLETITUD: GRAMMY ===
        Data Type  Null Count  Null Percentage (%)
nominee    object           6             0.124740
artist     object        1840            38.253638
workers    object        2190            45.530146
img        object        1367            28.419958


In [6]:
print("=== 2. UNICIDAD Y PATRONES DE DUPLICADOS ===")

# Spotify: Evaluación de track_id como llave candidata
unique_tracks = spotify_df['track_id'].nunique()
total_tracks = len(spotify_df)
track_dups = spotify_df.duplicated(subset=['track_id']).sum()

print(f"Spotify - Total de filas: {total_tracks:,}")
print(f"Spotify - 'track_id' únicos: {unique_tracks:,}")
print(f"Spotify - 'track_id' duplicados: {track_dups:,} ({(track_dups/total_tracks)*100:.2f}%)")

# Patrón de duplicación en Spotify (ejemplo de por qué se repiten)
sample_dup_id = spotify_df[spotify_df['track_id'].duplicated()]['track_id'].iloc[0]
print("\nEjemplo de patrón de duplicación en Spotify para un mismo 'track_id':")
print(spotify_df[spotify_df['track_id'] == sample_dup_id][['track_id', 'track_name', 'artists', 'track_genre', 'popularity']])

print("\n" + "="*60 + "\n")

# Grammy: Evaluación de clave compuesta (year, category, nominee)
grammy_key_dups = grammy_df.duplicated(subset=['year', 'category', 'nominee']).sum()
print(f"Grammy - Duplicados por clave compuesta (year, category, nominee): {grammy_key_dups}")

=== 2. UNICIDAD Y PATRONES DE DUPLICADOS ===
Spotify - Total de filas: 114,000
Spotify - 'track_id' únicos: 89,741
Spotify - 'track_id' duplicados: 24,259 (21.28%)

Ejemplo de patrón de duplicación en Spotify para un mismo 'track_id':
                    track_id        track_name                 artists  \
1874  0CDucx9lKxuCZplLXUz0iX  Song for Rollins  Buena Onda Reggae Club   
1925  0CDucx9lKxuCZplLXUz0iX  Song for Rollins  Buena Onda Reggae Club   

     track_genre  popularity  
1874    afrobeat          16  
1925    afrobeat          16  


Grammy - Duplicados por clave compuesta (year, category, nominee): 0


In [7]:
print("=== 3. CONTENIDO CATEGÓRICO Y NUMÉRICO ===")

print("\n--- SPOTIFY: Resumen Estadístico Numérico ---")
print(spotify_df[['popularity', 'duration_ms', 'danceability', 'energy', 'tempo']].describe().T[['min', 'mean', 'max', 'std']])

print("\n--- SPOTIFY: Análisis Categórico ---")
print(f"Cantidad de géneros distintos: {spotify_df['track_genre'].nunique()}")
print(f"Cantidad de artistas distintos: {spotify_df['artists'].nunique()}")

print("\n--- GRAMMY: Resumen Estadístico y Categórico ---")
print(f"Rango de años de premiación: Mínimo {grammy_df['year'].min()} - Máximo {grammy_df['year'].max()}")
print(f"Cantidad de categorías de premios distintas: {grammy_df['category'].nunique()}")
print(f"Distribución de la variable 'winner':")
print(grammy_df['winner'].value_counts(dropna=False))

=== 3. CONTENIDO CATEGÓRICO Y NUMÉRICO ===

--- SPOTIFY: Resumen Estadístico Numérico ---
              min           mean          max            std
popularity    0.0      33.238535      100.000      22.305078
duration_ms   0.0  228029.153114  5237295.000  107297.712645
danceability  0.0       0.566800        0.985       0.173542
energy        0.0       0.641383        1.000       0.251529
tempo         0.0     122.147837      243.372      29.978197

--- SPOTIFY: Análisis Categórico ---
Cantidad de géneros distintos: 114
Cantidad de artistas distintos: 31437

--- GRAMMY: Resumen Estadístico y Categórico ---
Rango de años de premiación: Mínimo 1958 - Máximo 2019
Cantidad de categorías de premios distintas: 638
Distribución de la variable 'winner':
winner
True    4810
Name: count, dtype: int64


In [8]:
print("=== 4. ANÁLISIS TEMPORAL (GRAMMY) ===")
year_coverage = grammy_df['year'].value_counts().sort_index()
print(f"Total de años cubiertos en Grammy: {len(year_coverage)} años")
print(f"Año inicial: {grammy_df['year'].min()} | Año final: {grammy_df['year'].max()}")
print("\nConteo de nominaciones por década:")
grammy_df['decade'] = (grammy_df['year'] // 10) * 10
print(grammy_df['decade'].value_counts().sort_index())

print("\n" + "="*60 + "\n")

print("=== 5. CONSISTENCIA CRUZADA ENTRE FUENTES (CROSS-SOURCE) ===")
# Normalización rápida para evaluar coincidencia exacta potencial de nombres de artista
spotify_artists = set(spotify_df['artists'].dropna().str.lower().str.strip())
grammy_artists = set(grammy_df['artist'].dropna().str.lower().str.strip())

common_artists = spotify_artists.intersection(grammy_artists)
print(f"Artistas únicos en Spotify: {len(spotify_artists):,}")
print(f"Artistas únicos en Grammy:  {len(grammy_artists):,}")
print(f"Artistas que coinciden exactamente entre ambas fuentes: {len(common_artists):,}")

=== 4. ANÁLISIS TEMPORAL (GRAMMY) ===
Total de años cubiertos en Grammy: 62 años
Año inicial: 1958 | Año final: 2019

Conteo de nominaciones por década:
decade
1950      63
1960     433
1970     487
1980     684
1990     866
2000    1067
2010    1210
Name: count, dtype: int64


=== 5. CONSISTENCIA CRUZADA ENTRE FUENTES (CROSS-SOURCE) ===
Artistas únicos en Spotify: 31,428
Artistas únicos en Grammy:  1,656
Artistas que coinciden exactamente entre ambas fuentes: 398


## 6. Matriz de Análisis de Riesgos de Calidad de Datos (Quality Risk Analysis)

A partir del perfilamiento de datos realizado sobre las fuentes crudas (`spotify_dataset.csv` y la tabla `raw_grammys` en PostgreSQL), se consolidan los riesgos de calidad identificados y su relación directa con los requerimientos de la ETL y el Data Warehouse:

| Dataset / Attribute | Profiling Evidence | Potential Quality Risk | Related Requirement |
| :--- | :--- | :--- | :--- |
| **Spotify / `track_id`** | 24,259 registros con `track_id` duplicado debido a múltiples clasificaciones de género (`track_genre`). | Duplicación de métricas de reproducción y cálculo erróneo de agregaciones numéricas en el Modelo Dimensional. | **Transformación**: Deduplicación por `track_id` conservando el registro con mayor `popularity`. |
| **Spotify / `Unnamed: 0`** | Columna que contiene un índice secuencial generado automáticamente durante la exportación del CSV. | Ruido en los datos (*Data noise*), consumo innecesario de almacenamiento y desalineación con el esquema destino. | **Limpieza**: Eliminación explícita de la columna en la capa de transformación. |
| **Spotify / `artists`, `album_name`, `track_name`** | 1 registro presenta valores nulos simultáneos en los tres atributos de metadata. | Imposibilidad de clasificar la entidad en las dimensiones de negocio (`Dim_Track`, `Dim_Artist`). | **Calidad/Limpieza**: Imputación de valores por defecto o filtrado de registros sin identificación básica. |
| **Spotify / `duration_ms`** | Presencia de duraciones iguales a $0\text{ ms}$ o extremadamente cortas. | Sesgo estadístico en el cálculo de promedios de duración por artista/género. | **Validación (GX)**: Regla de rango de duración $\ge 0\text{ ms}$ y filtrado en la capa de transformación. |
| **Grammy / `artist`** | 1,840 valores nulos en el atributo de artista nominado. | Pérdida de integridad referencial al intentar enlazar nominaciones históricas con la dimensión de Artista. | **Transformación**: Imputación controlada con la etiqueta `'Varios / No especificado'` (categorías de álbumes/compositores). |
| **Grammy / `workers`, `img`** | 2,190 y 1,367 valores nulos respectivamente en información complementaria. | Errores de carga o excepciones en el pipeline al insertar nulos en campos de texto. | **Transformación**: Reemplazo de nulos por cadenas vacías `''`. |
| **Cross-Source / Artistas y Canciones** | Diferencias ortográficas, caracteres especiales y variaciones de mayúsculas/minúsculas entre ambas fuentes. | Fallos en la integración/cruce (*joins*) entre los premios Grammy y el catálogo de Spotify. | **Estandarización**: Limpieza de cadenas (`strip`, conversión de caso, eliminación de caracteres de control) previo a la carga. |